# Laboratorio de FM: análisis espectral, PSD y ruido

Este cuaderno desarrolla los códigos a partir del modelo matemático del PDF. La secuencia va desde la señal moduladora finita hasta la estimación espectral con ruido y redes DFT de pesos fijos.

La idea central del documento es que una señal FM se puede escribir como

$$
x(t)=A_c\cos(\omega_ct+\phi(t)),
$$

donde la fase depende de la integral de la señal moduladora:

$$
\phi(t)=2\pi k_f\int m(t)\,dt.
$$

Cuando \(m(t)\) es periódica, su serie de Fourier produce desviaciones de fase por armónicos. En WBFM, esas desviaciones generan bandas laterales alrededor de la portadora y la potencia se distribuye entre componentes espectrales. En presencia de ruido blanco gaussiano, la PSD observada se interpreta como suma de la PSD de la señal y un piso de ruido.


## 1. Señal de prueba y modulación FM

El PDF parte de un modelo WBFM donde la señal moduladora periódica se expresa mediante una serie trigonométrica:

$$
m(t)=\sum_{n=1}^{\infty}A_n\cos(n\omega_mt+\theta_n).
$$

En este código la señal moduladora usada es una versión finita con armónicos impares:

$$
x_0(t)=\sum_{k=0}^{K}\frac{(-1)^k}{2k+1}\cos((2k+1)\omega_0t).
$$

Con \(T=1\,s\), se cumple \(f_0=1/T\) y \(\omega_0=2\pi/T\). La portadora se fija como

$$
\omega_c=100\omega_0,\qquad f_c=\frac{\omega_c}{2\pi}.
$$

Según el PDF, la desviación de fase de una señal FM se obtiene integrando la moduladora:

$$
\phi(t)=2\pi k_f\int x_0(t)\,dt.
$$

Como cada término de \(x_0(t)\) es un coseno, su integral produce un seno dividido por la frecuencia angular correspondiente. Por eso en el código aparece el factor

$$
\frac{(-1)^k}{(2k+1)^2\omega_0}.
$$

Finalmente se construye la señal modulada:

$$
x_{FM}(t)=A_c\cos(\omega_ct+\phi(t)).
$$


In [ ]:
import torch
import matplotlib.pyplot as plt

# ============================================================
# 1. PARÁMETROS EXIGIDOS POR EL PROFESOR
# ============================================================

K = 5

T = 1.0                    # Periodo fundamental [s]
w0 = 2 * torch.pi / T      # w0 = 2pi/T [rad/s]
f0 = 1 / T                 # f0 = 1/T [Hz]

wc = 100 * w0              # wc = 100*w0 [rad/s]
fc = wc / (2 * torch.pi)   # Frecuencia portadora [Hz]

Ac = 1.0                   # Amplitud de la portadora
kf = 25.0                  # Sensibilidad de frecuencia [Hz/unidad]

fs = 5000.0                # Frecuencia de muestreo [Hz]
Tsim = 10.0                # Tiempo de simulación [s]

N = int(fs * Tsim)
t = torch.arange(N) / fs

print("Parámetros:")
print(f"T  = {T:.2f} s")
print(f"f0 = {f0:.2f} Hz")
print(f"w0 = {w0:.2f} rad/s")
print(f"fc = {fc:.2f} Hz")
print(f"wc = {wc:.2f} rad/s")
print(f"wc / w0 = {wc / w0:.2f}")

# ============================================================
# 2. SEÑAL SIN MODULAR x0(t)
# ============================================================

def x0_signal(t, K, w0):
    """
    x0(t) = sum_{k=0}^{K} (-1)^k * 1/(2k+1)
            * cos((2k+1)w0 t)
    """

    x0 = torch.zeros_like(t)

    for k in range(K + 1):
        harmonic = 2 * k + 1
        coef = ((-1) ** k) / harmonic
        x0 += coef * torch.cos(harmonic * w0 * t)

    return x0


x0 = x0_signal(t, K, w0)

# ============================================================
# 3. FASE FM
# phi(t) = 2*pi*kf * integral{x0(t)}
# ============================================================

phi = torch.zeros_like(t)

for k in range(K + 1):
    harmonic = 2 * k + 1
    coef_phi = ((-1) ** k) / ((harmonic ** 2) * w0)
    phi += coef_phi * torch.sin(harmonic * w0 * t)

phi = 2 * torch.pi * kf * phi

# ============================================================
# 4. SEÑAL MODULADA FM
# ============================================================

x_fm = Ac * torch.cos(wc * t + phi)

# ============================================================
# 5. FUNCIÓN PARA CALCULAR ESPECTRO NORMALIZADO EN dB
# ============================================================

def spectrum_db(signal, fs):
    N = len(signal)

    X = torch.fft.fftshift(torch.fft.fft(signal))
    f = torch.fft.fftshift(torch.fft.fftfreq(N, d=1/fs))

    spectrum = torch.abs(X)
    spectrum = spectrum / torch.max(spectrum)

    spectrum_dB = 20 * torch.log10(spectrum + 1e-12)

    return f, spectrum_dB


f_x0, X0_dB = spectrum_db(x0, fs)
f_fm, XFM_dB = spectrum_db(x_fm, fs)

# ============================================================
# 6. FRECUENCIAS DE REFERENCIA EN EL EJE X
# ============================================================

harmonics = [2 * k + 1 for k in range(K + 1)]

xticks_x0 = sorted(
    [-h * f0 for h in harmonics] +
    [0] +
    [h * f0 for h in harmonics]
)

xticks_fm = sorted(
    [fc - h * f0 for h in harmonics] +
    [fc] +
    [fc + h * f0 for h in harmonics]
)

# ============================================================
# 7. ESPECTRO DE LA SEÑAL SIN MODULAR Y MODULADA
# ============================================================

plt.figure(figsize=(15, 5))

# ------------------------------------------------------------
# Espectro de x0(t)
# ------------------------------------------------------------

plt.subplot(1, 2, 1)
plt.plot(f_x0, X0_dB)
plt.title(r"Espectro de la señal sin modular $x_0(t)$")
plt.xlabel("Frecuencia [Hz]")
plt.ylabel("Magnitud normalizada [dB]")
plt.grid(True)

plt.xlim(-13 * f0, 13 * f0)
plt.ylim(-80, 5)

plt.xticks(xticks_x0)

# ------------------------------------------------------------
# Espectro de la señal modulada FM
# ------------------------------------------------------------

plt.subplot(1, 2, 2)
plt.plot(f_fm, XFM_dB)
plt.title(r"Espectro de la señal modulada FM")
plt.xlabel("Frecuencia [Hz]")
plt.ylabel("Magnitud normalizada [dB]")
plt.grid(True)

plt.xlim(fc - 20, fc + 20)
plt.ylim(-80, 5)

plt.xticks(xticks_fm, rotation=45)

plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'torch'

## 2. Espectro analítico de $x_0(t)$ y espectro numérico de FM

El PDF muestra que una señal FM genera bandas laterales alrededor de la portadora. Para la señal moduladora de este código, antes de modular, el contenido espectral es más simple: solo hay armónicos impares.

Si

$$
A\cos(n\omega_0t)
$$

aparece en la señal, entonces su espectro bilateral contiene componentes en

$$
\omega=n\omega_0 \quad \text{y} \quad \omega=-n\omega_0.
$$

En este caso:

$$
n=2k+1,\qquad A=\frac{(-1)^k}{2k+1}.
$$

El código calcula el espectro de \(x_0(t)\) de forma analítica usando esas líneas discretas, y calcula el espectro de la señal FM mediante FFT. Esto concuerda con el PDF: la FM distribuye energía en componentes alrededor de \(f_c\), mientras que la moduladora finita tiene líneas exactas en sus armónicos.


In [ ]:
import torch
import matplotlib.pyplot as plt

# ============================================================
# 1. PARÁMETROS EXIGIDOS
# ============================================================

K = 5

T = 1.0
w0 = 2 * torch.pi / T
f0 = 1 / T

wc = 100 * w0
fc = wc / (2 * torch.pi)

Ac = 1.0
kf = 25.0

fs = 5000.0
Tsim = 10.0

N = int(fs * Tsim)
t = torch.arange(N) / fs

print("Parámetros:")
print(f"T  = {T:.2f} s")
print(f"f0 = {f0:.2f} Hz")
print(f"w0 = {w0:.2f} rad/s")
print(f"fc = {fc:.2f} Hz")
print(f"wc = {wc:.2f} rad/s")
print(f"wc / w0 = {wc / w0:.2f}")

# ============================================================
# 2. SEÑAL x0(t)
# x0(t) = sum_{k=0}^{K} (-1)^k/(2k+1) cos((2k+1)w0t)
# ============================================================

x0 = torch.zeros_like(t)

for k in range(K + 1):
    n = 2 * k + 1
    coef = ((-1) ** k) / n
    x0 += coef * torch.cos(n * w0 * t)

# ============================================================
# 3. FASE FM
# phi(t) = 2*pi*kf * integral{x0(t)}
# ============================================================

phi = torch.zeros_like(t)

for k in range(K + 1):
    n = 2 * k + 1
    coef_phi = ((-1) ** k) / ((n ** 2) * w0)
    phi += coef_phi * torch.sin(n * w0 * t)

phi = 2 * torch.pi * kf * phi

# ============================================================
# 4. SEÑAL MODULADA FM
# ============================================================

x_fm = Ac * torch.cos(wc * t + phi)

# ============================================================
# 5. FUNCIÓN PARA ESPECTRO FFT NORMALIZADO EN dB
# ============================================================

def spectrum_db(signal, fs):
    N = len(signal)

    X = torch.fft.fftshift(torch.fft.fft(signal))
    f = torch.fft.fftshift(torch.fft.fftfreq(N, d=1/fs))

    mag = torch.abs(X)
    mag = mag / torch.max(mag)

    mag_db = 20 * torch.log10(mag + 1e-12)

    return f, mag_db


f_fm, XFM_dB = spectrum_db(x_fm, fs)

# ============================================================
# 6. ESPECTRO ANALÍTICO EN dB DE x0(t)
# ============================================================

harmonics = []
amplitudes = []

for k in range(K + 1):
    n = 2 * k + 1

    # Frecuencias normalizadas: ±n = ±(2k+1)w0 / w0
    harmonics.append(-n)
    harmonics.append(n)

    # Para A cos(nw0t), aparecen dos componentes A/2
    A = abs(((-1) ** k) / n)
    amplitudes.append(A / 2)
    amplitudes.append(A / 2)

harmonics = torch.tensor(harmonics, dtype=torch.float32)
amplitudes = torch.tensor(amplitudes, dtype=torch.float32)

# Normalización
amplitudes_norm = amplitudes / torch.max(amplitudes)

# Magnitud normalizada en dB
X0_dB = 20 * torch.log10(amplitudes_norm + 1e-12)

# Ordenar de izquierda a derecha
idx = torch.argsort(harmonics)
harmonics = harmonics[idx]
X0_dB = X0_dB[idx]

# ============================================================
# 7. GRÁFICAS
# ============================================================

plt.figure(figsize=(15, 5))

# ------------------------------------------------------------
# Espectro analítico de x0(t) en dB
# Corregido: los armónicos suben desde el piso de la gráfica
# ------------------------------------------------------------

plt.subplot(1, 2, 1)

ymin_x0 = -25
ymax_x0 = 2

# Dibujar líneas desde ymin_x0 hasta el valor en dB
plt.vlines(
    harmonics.numpy(),
    ymin=ymin_x0,
    ymax=X0_dB.numpy(),
    linewidth=1.8
)

# Dibujar puntos en la punta de cada armónico
plt.plot(
    harmonics.numpy(),
    X0_dB.numpy(),
    "o"
)

plt.title(r"Espectro de la señal sin modular $x_0(t)$ con $K=5$")
plt.xlabel(r"Frecuencia normalizada $\omega/\omega_0$")
plt.ylabel("Magnitud normalizada [dB]")
plt.grid(True)

plt.xlim(-12, 12)
plt.ylim(ymin_x0, ymax_x0)

xticks = [-11, -9, -7, -5, -3, -1, 0, 1, 3, 5, 7, 9, 11]
xtick_labels = [
    r"$-11\omega_0$", r"$-9\omega_0$", r"$-7\omega_0$",
    r"$-5\omega_0$", r"$-3\omega_0$", r"$-\omega_0$",
    r"$0$",
    r"$\omega_0$", r"$3\omega_0$", r"$5\omega_0$",
    r"$7\omega_0$", r"$9\omega_0$", r"$11\omega_0$"
]

plt.xticks(xticks, xtick_labels, rotation=45)

# ------------------------------------------------------------
# Espectro de la señal modulada FM
# ------------------------------------------------------------

plt.subplot(1, 2, 2)

plt.plot(f_fm.numpy(), XFM_dB.numpy())

plt.title(r"Espectro de la señal modulada FM")
plt.xlabel("Frecuencia [Hz]")
plt.ylabel("Magnitud normalizada [dB]")
plt.grid(True)

plt.xlim(fc - 40, fc + 40)
plt.ylim(-80, 5)

plt.tight_layout()
plt.show()

## 3. Visualización completa en tiempo y frecuencia

Este bloque reúne en una sola figura tres partes del modelo del PDF:

$$
x_0(t),\qquad X_0(f),\qquad x_{FM}(t).
$$

La señal \(x_0(t)\) corresponde a la suma finita de armónicos impares. Su máximo armónico para \(K=5\) es

$$
(2K+1)f_0=11f_0.
$$

La señal FM se obtiene con

$$
x_{FM}(t)=A_c\cos(\omega_ct+\phi(t)),
$$

donde

$$
\phi(t)=2\pi k_f\int x_0(t)\,dt.
$$

El PDF explica que, al aparecer la moduladora dentro de la fase, el espectro FM no queda limitado a una sola línea en \(f_c\), sino que aparecen bandas laterales. Por eso el código grafica el espectro alrededor de la portadora.


In [ ]:
import torch
import matplotlib.pyplot as plt

# ============================================================
# 1. PARÁMETROS
# ============================================================

K = 5

f0 = 5.0                  # Frecuencia fundamental de x0(t) [Hz]
w0 = 2 * torch.pi * f0    # Frecuencia angular fundamental

fc = 80.0                 # Frecuencia portadora [Hz]
wc = 2 * torch.pi * fc

Ac = 1.0                  # Amplitud de la portadora
kf = 25.0                 # Sensibilidad de frecuencia [Hz/unidad]

fs = 5000.0               # Frecuencia de muestreo [Hz]
Tsim = 4.0                # Tiempo de simulación [s]

N = int(fs * Tsim)
t = torch.arange(N) / fs

print("Frecuencia central:")
print(f"fc = {fc:.2f} Hz")
print(f"wc / w0 = {wc / w0:.2f}")

# ============================================================
# 2. IMPLEMENTACIÓN DE x0(t)
# ============================================================

def x0_signal(t, K, w0):
    """
    Implementa:

        x0(t) = sum_{k=0}^{K} (-1)^k * 1/(2k+1)
                * cos((2k+1)w0 t)

    usando PyTorch.
    """

    k = torch.arange(0, K + 1, dtype=t.dtype, device=t.device)

    harmonics = 2 * k + 1

    coefficients = (-1.0) ** k / harmonics

    terms = coefficients[:, None] * torch.cos(
        harmonics[:, None] * w0 * t[None, :]
    )

    x0 = torch.sum(terms, dim=0)

    return x0


x0 = x0_signal(t, K, w0)

# ============================================================
# 3. FASE FM
# phi(t) = 2*pi*kf * integral{x0(t)}
# ============================================================

phi = torch.zeros_like(t)

for k in range(K + 1):
    harmonic = 2 * k + 1

    coef_phi = ((-1) ** k) / ((harmonic ** 2) * w0)

    phi += coef_phi * torch.sin(harmonic * w0 * t)

phi = 2 * torch.pi * kf * phi

# ============================================================
# 4. SEÑAL MODULADA FM
# ============================================================

x_fm = Ac * torch.cos(wc * t + phi)

# ============================================================
# 5. FUNCIÓN PARA CALCULAR ESPECTRO NORMALIZADO EN dB
# ============================================================

def spectrum_db(signal, fs):
    N = len(signal)

    X = torch.fft.fftshift(torch.fft.fft(signal))
    f = torch.fft.fftshift(torch.fft.fftfreq(N, d=1/fs))

    spectrum = torch.abs(X)
    spectrum = spectrum / torch.max(spectrum)

    spectrum_dB = 20 * torch.log10(spectrum + 1e-12)

    return f, spectrum_dB


# Espectro de la señal sin modular
f_x0, X0_dB = spectrum_db(x0, fs)

# Espectro de la señal modulada
f_fm, XFM_dB = spectrum_db(x_fm, fs)

# ============================================================
# 6. UNA SOLA IMAGEN CON LAS 3 GRÁFICAS
# ============================================================

plt.figure(figsize=(15, 10))

# ------------------------------------------------------------
# Gráfica 1: señal x0(t) en el tiempo
# ------------------------------------------------------------

plt.subplot(3, 1, 1)
plt.plot(t.numpy(), x0.numpy(), label=r"$x_0(t)$ con $K=5$")
plt.title(r"Señal sin modular $x_0(t)$ en el tiempo")
plt.xlabel("Tiempo [s]")
plt.ylabel("Amplitud")
plt.grid(True)
plt.legend()

# Mostrar solo una parte para verla mejor
plt.xlim(0, 1)

# ------------------------------------------------------------
# Gráfica 2: espectro de x0(t)
# ------------------------------------------------------------

plt.subplot(3, 1, 2)
plt.plot(f_x0.numpy() / f0, X0_dB.numpy())
plt.title(r"Espectro de la señal sin modular $x_0(t)$")
plt.xlabel(r"Frecuencia normalizada $f/f_0$")
plt.ylabel("Magnitud normalizada [dB]")
plt.grid(True)

plt.xlim(-13, 13)
plt.ylim(-80, 5)

# Marcar armónicos impares
for k in range(K + 1):
    harmonic = 2 * k + 1
    plt.plot(harmonic, 0, "o")
    plt.plot(-harmonic, 0, "o")

# ------------------------------------------------------------
# Gráfica 3: espectro de la señal modulada FM
# ------------------------------------------------------------

plt.subplot(3, 1, 3)
plt.plot(f_fm.numpy(), XFM_dB.numpy())
plt.title(r"Espectro de la señal modulada FM")
plt.xlabel("Frecuencia [Hz]")
plt.ylabel("Magnitud normalizada [dB]")
plt.grid(True)

plt.xlim(fc - 50, fc + 50)
plt.ylim(-80, 5)

plt.tight_layout()
plt.show()

## 4. PSD teórica y cálculo matricial de la DFT

El PDF indica que la potencia de una señal FM se reparte entre componentes espectrales, pero la potencia total se conserva. En este código se analiza primero la potencia espectral de la señal determinística \(x_0(t)\).

Para un término

$$
A\cos(2\pi ft),
$$

el espectro bilateral reparte la amplitud en dos líneas simétricas. Por eso la potencia asociada a cada línea es proporcional a

$$
\left(\frac{A}{2}\right)^2.
$$

La segunda parte del código implementa la DFT como una red con pesos fijos:

$$
X[k]=\sum_{n=0}^{N-1}x[n]e^{-j2\pi kn/N}.
$$

Separando parte real e imaginaria:

$$
\Re\{X[k]\}=\sum_n x[n]\cos\left(\frac{2\pi kn}{N}\right),
$$

$$
\Im\{X[k]\}=-\sum_n x[n]\sin\left(\frac{2\pi kn}{N}\right).
$$

La red no aprende: solo calcula la DFT mediante productos matriciales y luego estima la PSD como

$$
|X[k]|^2=\Re\{X[k]\}^2+\Im\{X[k]\}^2.
$$


In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import time

# ============================================================
# 1. PARÁMETROS EXIGIDOS
# ============================================================

K = 5

T = 1.0
w0 = 2 * torch.pi / T
f0 = 1 / T

fs = 1024.0        # Frecuencia de muestreo [Hz]
Tsim = 1.0         # Tiempo de simulación [s]
N = int(fs * Tsim)

t = torch.arange(N) / fs

print("Parámetros:")
print(f"T  = {T:.2f} s")
print(f"f0 = {f0:.2f} Hz")
print(f"w0 = {w0:.2f} rad/s")
print(f"fs = {fs:.2f} Hz")
print(f"N  = {N}")

# ============================================================
# 2. SEÑAL x0(t) CON K = 5
# ============================================================

x0 = torch.zeros_like(t)

for k in range(K + 1):
    n = 2 * k + 1
    coef = ((-1) ** k) / n
    x0 += coef * torch.cos(n * w0 * t)

# ============================================================
# 3. PSD TEÓRICA
# ============================================================

start = time.perf_counter()

freqs = torch.fft.fftshift(torch.fft.fftfreq(N, d=1/fs))

PSD_teorica = torch.zeros(N)

for k in range(K + 1):
    n = 2 * k + 1

    freq_pos = n * f0
    freq_neg = -n * f0

    A = abs(((-1) ** k) / n)

    # Para A cos(2πft), la magnitud espectral tiene A/2 en ±f.
    # La potencia es proporcional a (A/2)^2.
    power = (A / 2) ** 2

    idx_pos = torch.argmin(torch.abs(freqs - freq_pos))
    idx_neg = torch.argmin(torch.abs(freqs - freq_neg))

    PSD_teorica[idx_pos] = power
    PSD_teorica[idx_neg] = power

PSD_teorica = PSD_teorica / torch.max(PSD_teorica)

end = time.perf_counter()
tiempo_psd_teorica = end - start

# ============================================================
# 4. RED DFT CON PESOS FIJOS
# ============================================================
# Esta red no se entrena.
# Sus pesos son los coeficientes de la DFT:
#
# X[k] = sum_n x[n] exp(-j 2πkn/N)
#
# Se implementa con dos matrices:
# una para la parte real y otra para la parte imaginaria.
# ============================================================

class DFTNet(nn.Module):
    def __init__(self, N):
        super().__init__()

        start = time.perf_counter()

        n = torch.arange(N).float()
        k = torch.arange(N).float().unsqueeze(1)

        angle = 2 * torch.pi * k * n / N

        W_real = torch.cos(angle)
        W_imag = -torch.sin(angle)

        self.register_buffer("W_real", W_real)
        self.register_buffer("W_imag", W_imag)

        end = time.perf_counter()
        self.tiempo_construccion_pesos = end - start

    def forward(self, x):
        X_real = self.W_real @ x
        X_imag = self.W_imag @ x

        PSD = X_real**2 + X_imag**2

        PSD = torch.fft.fftshift(PSD)
        PSD = PSD / torch.max(PSD)

        return PSD

# Construcción de pesos de la red DFT
dft_net = DFTNet(N)
tiempo_construccion_pesos = dft_net.tiempo_construccion_pesos

# ============================================================
# 5. INFERENCIA DE PSD USANDO LA RED DFT
# ============================================================

num_repeticiones = 100

# Primera inferencia
with torch.no_grad():
    PSD_nn = dft_net(x0)

# Medición promedio de inferencia
start = time.perf_counter()

with torch.no_grad():
    for _ in range(num_repeticiones):
        PSD_nn = dft_net(x0)

end = time.perf_counter()

tiempo_inferencia_psd_nn = (end - start) / num_repeticiones

# ============================================================
# 6. RESULTADOS DE TIEMPO
# ============================================================

print("\nTiempos de cálculo:")
print(f"Tiempo PSD teórica                      = {tiempo_psd_teorica:.8f} s")
print(f"Tiempo construcción pesos red DFT       = {tiempo_construccion_pesos:.8f} s")
print(f"Tiempo inferencia PSD red neuronal DFT  = {tiempo_inferencia_psd_nn:.8f} s")

# ============================================================
# 7. GRÁFICA SUPERPUESTA
# ============================================================

plt.figure(figsize=(12, 5))

plt.stem(
    freqs.numpy(),
    PSD_teorica.numpy(),
    linefmt="C0-",
    markerfmt="C0o",
    basefmt=" ",
    label="PSD teórica"
)

plt.plot(
    freqs.numpy(),
    PSD_nn.numpy(),
    "C1",
    linewidth=2,
    label="PSD estimada con red DFT"
)

plt.title(r"Comparación superpuesta: PSD teórica vs PSD por red DFT, $K=5$")
plt.xlabel("Frecuencia [Hz]")
plt.ylabel("PSD normalizada")
plt.grid(True)
plt.xlim(-15, 15)
plt.ylim(-0.05, 1.1)
plt.legend()

plt.tight_layout()
plt.show()

## 5. PSD con ruido gaussiano para \(K=5\)

El PDF modela una señal recibida como la suma de señal y ruido:

$$
y(t)=x(t)+n(t),
$$

donde \(n(t)\) es ruido gaussiano blanco de media cero. En este código se aplica ese mismo modelo sobre \(x_0(t)\):

$$
y(t)=x_0(t)+n(t),\qquad n(t)\sim\mathcal{N}(0,\sigma_n^2).
$$

La PSD se estima con el periodograma bilateral:

$$
P_{xx}(f)=\frac{|X(f)|^2}{Nf_s}.
$$

Para cada varianza se generan \(M=100\) realizaciones ruidosas y se promedian sus periodogramas. Esto sigue la idea del PDF: la PSD observada se puede interpretar como

$$
S_y(f)=S_x(f)+S_n(f).
$$

Al aumentar \(\sigma_n^2\), el piso espectral del ruido sube y las líneas armónicas de \(x_0(t)\) pierden contraste.


In [ ]:
import torch
import matplotlib.pyplot as plt

# ============================================================
# 1. PARÁMETROS DE LA SEÑAL
# ============================================================

K = 5

T = 1.0
w0 = 2 * torch.pi / T
f0 = 1 / T

fs = 5000.0
Tsim = 10.0

N = int(fs * Tsim)
t = torch.arange(N) / fs

# ============================================================
# 2. PARÁMETROS DE REALIZACIONES
# ============================================================

M = 100
media_ruido = 0.0

varianzas = [0.1, 1.0, 5.0]

print("Parámetros:")
print(f"K = {K}")
print(f"T = {T:.2f} s")
print(f"f0 = {f0:.2f} Hz")
print(f"w0 = {w0:.2f} rad/s")
print(f"fs = {fs:.2f} Hz")
print(f"N = {N}")
print(f"Número de realizaciones = {M}")
print(f"Media del ruido = {media_ruido}")

# ============================================================
# 3. SEÑAL x0(t)
# ============================================================

def x0_signal(t, K, w0):
    """
    x0(t) = sum_{k=0}^{K} (-1)^k * 1/(2k+1)
            * cos((2k+1)w0 t)
    """

    x0 = torch.zeros_like(t)

    for k in range(K + 1):
        n = 2 * k + 1
        coef = ((-1) ** k) / n
        x0 += coef * torch.cos(n * w0 * t)

    return x0


x0 = x0_signal(t, K, w0)

# ============================================================
# 4. FUNCIÓN PERIODOGRAMA
# ============================================================

def periodograma_bilateral(x, fs):
    """
    Estima la PSD bilateral mediante periodograma:

        Pxx(f) = |X(f)|^2 / (N fs)

    Si x tiene dimensión (M, N), calcula M periodogramas.
    Si x tiene dimensión (N,), calcula un solo periodograma.
    """

    if x.ndim == 1:
        N = x.shape[0]

        X = torch.fft.fftshift(torch.fft.fft(x))
        f = torch.fft.fftshift(torch.fft.fftfreq(N, d=1/fs))

        Pxx = (torch.abs(X) ** 2) / (N * fs)

        return f, Pxx

    elif x.ndim == 2:
        N = x.shape[1]

        X = torch.fft.fftshift(torch.fft.fft(x, dim=1), dim=1)
        f = torch.fft.fftshift(torch.fft.fftfreq(N, d=1/fs))

        Pxx = (torch.abs(X) ** 2) / (N * fs)

        return f, Pxx


# ============================================================
# 5. PSD DE LA SEÑAL LIMPIA
# ============================================================

freqs, PSD_limpia = periodograma_bilateral(x0, fs)

referencia = torch.max(PSD_limpia)

PSD_limpia_norm = PSD_limpia / referencia
PSD_limpia_dB = 10 * torch.log10(PSD_limpia_norm + 1e-12)

# ============================================================
# 6. EJE X EN ARMÓNICOS
# ============================================================

armonicos = [-11, -9, -7, -5, -3, -1, 0, 1, 3, 5, 7, 9, 11]

labels_armonicos = [
    r"$-11\omega_0$", r"$-9\omega_0$", r"$-7\omega_0$",
    r"$-5\omega_0$", r"$-3\omega_0$", r"$-\omega_0$",
    r"$0$",
    r"$\omega_0$", r"$3\omega_0$", r"$5\omega_0$",
    r"$7\omega_0$", r"$9\omega_0$", r"$11\omega_0$"
]

# ============================================================
# 7. FIGURA CON TRES GRÁFICAS EN UNA FILA
# ============================================================

fig, axs = plt.subplots(1, 3, figsize=(21, 5), sharey=True)

for i, var_ruido in enumerate(varianzas):

    sigma_ruido = torch.sqrt(torch.tensor(var_ruido))

    # --------------------------------------------------------
    # Generar 100 realizaciones con ruido
    # --------------------------------------------------------

    x0_matrix = x0.unsqueeze(0).repeat(M, 1)

    ruido = media_ruido + sigma_ruido * torch.randn(M, N)

    x_ruidosas = x0_matrix + ruido

    # --------------------------------------------------------
    # Verificar SNR obtenido
    # --------------------------------------------------------

    P_signal = torch.mean(x0**2)
    P_noise = torch.mean(ruido**2)

    SNR_real = P_signal / P_noise
    SNR_real_dB = 10 * torch.log10(SNR_real)

    print("\nCaso:")
    print(f"Varianza del ruido = {var_ruido}")
    print(f"Potencia promedio de x0(t) = {P_signal.item():.6f}")
    print(f"Potencia promedio del ruido = {P_noise.item():.6f}")
    print(f"SNR obtenido = {SNR_real_dB.item():.2f} dB")

    # --------------------------------------------------------
    # PSD promedio con ruido
    # --------------------------------------------------------

    _, periodogramas_ruidosos = periodograma_bilateral(x_ruidosas, fs)

    PSD_ruidosa_promedio = torch.mean(periodogramas_ruidosos, dim=0)

    PSD_ruidosa_norm = PSD_ruidosa_promedio / referencia
    PSD_ruidosa_dB = 10 * torch.log10(PSD_ruidosa_norm + 1e-12)

    # --------------------------------------------------------
    # Graficar en el subplot correspondiente
    # --------------------------------------------------------

    axs[i].plot(
        freqs / f0,
        PSD_limpia_dB,
        linewidth=2,
        label="PSD sin ruido"
    )

    axs[i].plot(
        freqs / f0,
        PSD_ruidosa_dB,
        linewidth=2,
        alpha=0.8,
        label="PSD con ruido"
    )

    axs[i].set_title(rf"$\sigma_n^2={var_ruido}$")
    axs[i].set_xlabel(r"Frecuencia normalizada $\omega/\omega_0$")
    axs[i].grid(True)

    axs[i].set_xlim(-12, 12)
    axs[i].set_ylim(-80, 10)

    axs[i].set_xticks(armonicos)
    axs[i].set_xticklabels(labels_armonicos, rotation=45)

    axs[i].legend()

axs[0].set_ylabel("PSD normalizada [dB]")

fig.suptitle(r"Comparación de PSD de $x_0(t)$ con $K=5$ para diferentes varianzas de ruido", fontsize=14)

plt.tight_layout()
plt.show()

## 6. Ancho de banda determinístico y ruido para \(K=9\)

El PDF señala que la señal de prueba contiene armónicos impares hasta el orden \(2K+1\). Por tanto, su máxima componente determinística es

$$
f_{\max}=(2K+1)f_0.
$$

Como el espectro bilateral incluye frecuencias positivas y negativas, el ancho de banda total usado en el código es

$$
BW=2(2K+1)\omega_0.
$$

Para \(K=9\):

$$
2K+1=19.
$$

Así, el código analiza una señal con más componentes espectrales que en el caso \(K=5\). Luego repite el modelo con ruido gaussiano para comparar cómo cambia la PSD ante diferentes varianzas.


In [ ]:
import torch
import matplotlib.pyplot as plt

# ============================================================
# 1. PARÁMETROS DE LA SEÑAL
# ============================================================

K = 9

T = 1.0
w0 = 2 * torch.pi / T
f0 = 1 / T

fs = 5000.0
Tsim = 10.0

N = int(fs * Tsim)
t = torch.arange(N) / fs

# ============================================================
# 2. PARÁMETROS DE REALIZACIONES
# ============================================================

M = 100
media_ruido = 0.0

varianzas = [0.1, 1.0, 5.0]

print("Parámetros:")
print(f"K = {K}")
print(f"T = {T:.2f} s")
print(f"f0 = {f0:.2f} Hz")
print(f"w0 = {w0:.2f} rad/s")
print(f"fs = {fs:.2f} Hz")
print(f"N = {N}")
print(f"Número de realizaciones = {M}")
print(f"Media del ruido = {media_ruido}")

# ============================================================
# 3. ANCHO DE BANDA TEÓRICO
# BW = 2(2K+1)w0
# ============================================================

BW_rad = 2 * (2*K + 1) * w0
BW_Hz = BW_rad / (2 * torch.pi)

print("\nAncho de banda teórico:")
print(f"BW = 2(2K+1)w0")
print(f"BW = 2(2({K})+1)w0")
print(f"BW = {BW_rad:.4f} rad/s")
print(f"BW = {BW_Hz:.4f} Hz")

# ============================================================
# 4. SEÑAL x0(t) CON K = 9
# ============================================================

def x0_signal(t, K, w0):
    """
    x0(t) = sum_{k=0}^{K} (-1)^k * 1/(2k+1)
            * cos((2k+1)w0 t)
    """

    x0 = torch.zeros_like(t)

    for k in range(K + 1):
        n = 2 * k + 1
        coef = ((-1) ** k) / n
        x0 += coef * torch.cos(n * w0 * t)

    return x0


x0 = x0_signal(t, K, w0)

# ============================================================
# 5. FUNCIÓN PERIODOGRAMA
# ============================================================

def periodograma_bilateral(x, fs):
    """
    Estima la PSD bilateral mediante periodograma:

        Pxx(f) = |X(f)|^2 / (N fs)

    Si x tiene dimensión (M, N), calcula M periodogramas.
    Si x tiene dimensión (N,), calcula un solo periodograma.
    """

    if x.ndim == 1:
        N = x.shape[0]

        X = torch.fft.fftshift(torch.fft.fft(x))
        f = torch.fft.fftshift(torch.fft.fftfreq(N, d=1/fs))

        Pxx = (torch.abs(X) ** 2) / (N * fs)

        return f, Pxx

    elif x.ndim == 2:
        N = x.shape[1]

        X = torch.fft.fftshift(torch.fft.fft(x, dim=1), dim=1)
        f = torch.fft.fftshift(torch.fft.fftfreq(N, d=1/fs))

        Pxx = (torch.abs(X) ** 2) / (N * fs)

        return f, Pxx


# ============================================================
# 6. PSD DE LA SEÑAL LIMPIA
# ============================================================

freqs, PSD_limpia = periodograma_bilateral(x0, fs)

referencia = torch.max(PSD_limpia)

PSD_limpia_norm = PSD_limpia / referencia
PSD_limpia_dB = 10 * torch.log10(PSD_limpia_norm + 1e-12)

# ============================================================
# 7. EJE X EN ARMÓNICOS PARA K = 9
# ============================================================

armonicos_pos = [2*k + 1 for k in range(K + 1)]
armonicos = [-h for h in reversed(armonicos_pos)] + [0] + armonicos_pos

labels_armonicos = []

for h in armonicos:
    if h == 0:
        labels_armonicos.append(r"$0$")
    elif h == -1:
        labels_armonicos.append(r"$-\omega_0$")
    elif h == 1:
        labels_armonicos.append(r"$\omega_0$")
    else:
        labels_armonicos.append(rf"${h}\omega_0$")

# ============================================================
# 8. FIGURA CON TRES GRÁFICAS EN UNA FILA
# ============================================================

fig, axs = plt.subplots(1, 3, figsize=(24, 5), sharey=True)

for i, var_ruido in enumerate(varianzas):

    sigma_ruido = torch.sqrt(torch.tensor(var_ruido))

    # --------------------------------------------------------
    # Generar 100 realizaciones con ruido
    # --------------------------------------------------------

    x0_matrix = x0.unsqueeze(0).repeat(M, 1)

    ruido = media_ruido + sigma_ruido * torch.randn(M, N)

    x_ruidosas = x0_matrix + ruido

    # --------------------------------------------------------
    # Verificar SNR obtenido
    # --------------------------------------------------------

    P_signal = torch.mean(x0**2)
    P_noise = torch.mean(ruido**2)

    SNR_real = P_signal / P_noise
    SNR_real_dB = 10 * torch.log10(SNR_real)

    print("\nCaso:")
    print(f"Varianza del ruido = {var_ruido}")
    print(f"Potencia promedio de x0(t) = {P_signal.item():.6f}")
    print(f"Potencia promedio del ruido = {P_noise.item():.6f}")
    print(f"SNR obtenido = {SNR_real_dB.item():.2f} dB")
    print(f"BW teórico = {BW_rad:.4f} rad/s = {BW_Hz:.4f} Hz")

    # --------------------------------------------------------
    # PSD promedio con ruido
    # --------------------------------------------------------

    _, periodogramas_ruidosos = periodograma_bilateral(x_ruidosas, fs)

    PSD_ruidosa_promedio = torch.mean(periodogramas_ruidosos, dim=0)

    PSD_ruidosa_norm = PSD_ruidosa_promedio / referencia
    PSD_ruidosa_dB = 10 * torch.log10(PSD_ruidosa_norm + 1e-12)

    # --------------------------------------------------------
    # Graficar
    # --------------------------------------------------------

    axs[i].plot(
        freqs / f0,
        PSD_limpia_dB,
        linewidth=2,
        label="PSD sin ruido"
    )

    axs[i].plot(
        freqs / f0,
        PSD_ruidosa_dB,
        linewidth=2,
        alpha=0.8,
        label="PSD con ruido"
    )

    axs[i].set_title(
        rf"$\sigma_n^2={var_ruido}$" + "\n" +
        rf"$BW=2(2K+1)\omega_0={BW_rad:.2f}$ rad/s"
    )

    axs[i].set_xlabel(r"Frecuencia normalizada $\omega/\omega_0$")
    axs[i].grid(True)

    axs[i].set_xlim(-(2*K + 3), (2*K + 3))
    axs[i].set_ylim(-80, 10)

    axs[i].set_xticks(armonicos)
    axs[i].set_xticklabels(labels_armonicos, rotation=45, fontsize=8)

    axs[i].legend()

axs[0].set_ylabel("PSD normalizada [dB]")

fig.suptitle(
    rf"Comparación de PSD de $x_0(t)$ con $K=9$ para diferentes varianzas de ruido",
    fontsize=14
)

plt.tight_layout()
plt.show()

## 7. Periodograma por FFT y por red DFT en presencia de ruido

El PDF propone estimar el ancho de banda de señales WBFM a partir de su PSD y menciona que una red puede aprender o explotar estructuras espectrales. En este código no se entrena una CNN; se construye una red lineal de pesos fijos para reproducir el cálculo del periodograma en una banda de interés.

La señal ruidosa es:

$$
y(t)=x_0(t)+n(t),\qquad n(t)\sim\mathcal{N}(0,5).
$$

El periodograma clásico se calcula como

$$
P_{yy}(f)=\frac{|Y(f)|^2}{Nf_s}.
$$

La red evalúa la DFT en frecuencias objetivo:

$$
X(f_m)=\sum_{n=0}^{N-1}x[n]e^{-j2\pi f_m n/f_s}.
$$

Sus dos capas representan:

$$
\cos\left(\frac{2\pi f_m n}{f_s}\right)
\quad \text{y} \quad
-\sin\left(\frac{2\pi f_m n}{f_s}\right).
$$

Por eso la salida de la red y la FFT deben coincidir numéricamente. El error medio absoluto impreso por el código verifica esa equivalencia.


In [ ]:
import torch
import torch.nn as nn
import math
import matplotlib.pyplot as plt

# ============================================================
# 1. PARÁMETROS DE LA SEÑAL
# ============================================================

K = 9

T = 1.0
w0 = 2 * torch.pi / T
f0 = 1 / T

fs = 5000.0
Tsim = 10.0

N = int(fs * Tsim)
t = torch.arange(N) / fs

M = 100

media_ruido = 0.0
var_ruido = 5.0
sigma_ruido = torch.sqrt(torch.tensor(var_ruido))

print("Parámetros:")
print(f"K = {K}")
print(f"T = {T:.2f} s")
print(f"f0 = {f0:.2f} Hz")
print(f"w0 = {w0:.4f} rad/s")
print(f"fs = {fs:.2f} Hz")
print(f"Tsim = {Tsim:.2f} s")
print(f"N = {N}")
print(f"Realizaciones = {M}")
print(f"Media ruido = {media_ruido}")
print(f"Varianza ruido = {var_ruido}")

# ============================================================
# 2. ANCHO DE BANDA TEÓRICO
# BW = 2(2K+1)w0
# ============================================================

BW_rad = 2 * (2*K + 1) * w0
BW_Hz = BW_rad / (2 * torch.pi)

print("\nAncho de banda teórico:")
print(f"BW = 2(2K+1)w0")
print(f"BW = {BW_rad:.4f} rad/s")
print(f"BW = {BW_Hz:.4f} Hz")

# ============================================================
# 3. SEÑAL LIMPIA x0(t)
# x0(t) = sum_{k=0}^{K} (-1)^k/(2k+1) cos((2k+1)w0t)
# ============================================================

def x0_signal(t, K, w0):
    x0 = torch.zeros_like(t)

    for k in range(K + 1):
        n = 2*k + 1
        coef = ((-1) ** k) / n
        x0 += coef * torch.cos(n * w0 * t)

    return x0


x0 = x0_signal(t, K, w0)

# ============================================================
# 4. GENERAR 100 REALIZACIONES CON RUIDO
# y(t) = x0(t) + n(t), n(t) ~ N(0, 5)
# ============================================================

x0_batch = x0.unsqueeze(0).repeat(M, 1)

ruido = media_ruido + sigma_ruido * torch.randn(M, N)

y_batch = x0_batch + ruido

# Una realización para graficar en tiempo si se desea
y_ejemplo = y_batch[0]

# ============================================================
# 5. PERIODOGRAMA CLÁSICO BILATERAL CON FFT
# ============================================================

def periodograma_bilateral_fft(x, fs):
    """
    Periodograma bilateral clásico:

        Pxx(f) = |X(f)|^2 / (N fs)

    Si x tiene forma [N], calcula un periodograma.
    Si x tiene forma [M, N], calcula M periodogramas.
    """

    if x.dim() == 1:
        N = x.shape[0]

        X = torch.fft.fftshift(torch.fft.fft(x))
        f = torch.fft.fftshift(torch.fft.fftfreq(N, d=1/fs))

        Pxx = (torch.abs(X) ** 2) / (N * fs)

        return f, Pxx

    elif x.dim() == 2:
        N = x.shape[1]

        X = torch.fft.fftshift(torch.fft.fft(x, dim=1), dim=1)
        f = torch.fft.fftshift(torch.fft.fftfreq(N, d=1/fs))

        Pxx = (torch.abs(X) ** 2) / (N * fs)

        return f, Pxx


freqs_full, Pxx_limpia_full = periodograma_bilateral_fft(x0, fs)
_, Pyy_batch_full = periodograma_bilateral_fft(y_batch, fs)

Pyy_promedio_full = torch.mean(Pyy_batch_full, dim=0)

# ============================================================
# 6. SELECCIONAR SOLO LA BANDA DE INTERÉS
# ============================================================

fmax = 22 * f0

mask = (freqs_full >= -fmax) & (freqs_full <= fmax)

freqs = freqs_full[mask]
Pxx_limpia = Pxx_limpia_full[mask]
Pyy_promedio = Pyy_promedio_full[mask]

# ============================================================
# 7. RED DFT PARA CALCULAR PSD POR PERIODOGRAMA
# ============================================================

class RedPSDPeriodograma(nn.Module):
    def __init__(self, N, fs, freqs_objetivo):
        super().__init__()

        self.N = N
        self.fs = fs

        self.freqs_objetivo = freqs_objetivo.float()
        self.Nf = len(freqs_objetivo)

        n = torch.arange(N).float()

        angulo = 2 * math.pi * torch.outer(self.freqs_objetivo, n) / fs

        W_real = torch.cos(angulo)
        W_imag = -torch.sin(angulo)

        self.capa_real = nn.Linear(N, self.Nf, bias=False)
        self.capa_imag = nn.Linear(N, self.Nf, bias=False)

        with torch.no_grad():
            self.capa_real.weight.copy_(W_real)
            self.capa_imag.weight.copy_(W_imag)

        for p in self.parameters():
            p.requires_grad = False

    def forward(self, x):
        if x.dim() == 1:
            x = x.unsqueeze(0)

        X_real = self.capa_real(x)
        X_imag = self.capa_imag(x)

        Pxx = X_real**2 + X_imag**2

        Pxx = Pxx / (self.N * self.fs)

        Pxx_promedio = torch.mean(Pxx, dim=0)

        return Pxx_promedio, Pxx


red_psd = RedPSDPeriodograma(N, fs, freqs)

Pyy_red_promedio, Pyy_red_individual = red_psd(y_batch)

Pxx_red_limpia, _ = red_psd(x0)

# ============================================================
# 8. NORMALIZACIÓN EN dB
# ============================================================

referencia = torch.max(Pxx_limpia)

def normalizar_dB(P, referencia):
    P_norm = P / referencia
    P_dB = 10 * torch.log10(P_norm + 1e-30)
    return P_dB


Pxx_limpia_dB = normalizar_dB(Pxx_limpia, referencia)
Pyy_promedio_dB = normalizar_dB(Pyy_promedio, referencia)
Pyy_red_dB = normalizar_dB(Pyy_red_promedio, referencia)
Pxx_red_limpia_dB = normalizar_dB(Pxx_red_limpia, referencia)

# ============================================================
# 9. ERROR ENTRE PERIODOGRAMA CLÁSICO Y RED DFT
# ============================================================

error_psd_ruido = torch.mean(torch.abs(Pyy_promedio - Pyy_red_promedio)).item()
error_psd_limpia = torch.mean(torch.abs(Pxx_limpia - Pxx_red_limpia)).item()

print("\n===== COMPARACIÓN PSD CLÁSICA VS RED DFT =====")
print(f"Error medio absoluto PSD ruidosa promedio = {error_psd_ruido:.6e}")
print(f"Error medio absoluto PSD limpia           = {error_psd_limpia:.6e}")

# ============================================================
# 10. EJE X EN ARMÓNICOS IMPARES
# ============================================================

armonicos_pos = [2*k + 1 for k in range(K + 1)]
armonicos = [-h for h in reversed(armonicos_pos)] + [0] + armonicos_pos

labels_armonicos = []

for h in armonicos:
    if h == 0:
        labels_armonicos.append(r"$0$")
    elif h == -1:
        labels_armonicos.append(r"$-\omega_0$")
    elif h == 1:
        labels_armonicos.append(r"$\omega_0$")
    else:
        labels_armonicos.append(rf"${h}\omega_0$")

# ============================================================
# 11. GRÁFICA EN EL DOMINIO DEL TIEMPO
# ============================================================

plt.figure(figsize=(12, 4))

plt.plot(
    t,
    x0,
    linewidth=2,
    label=r"Señal limpia $x_0(t)$"
)

plt.plot(
    t,
    y_ejemplo,
    alpha=0.7,
    label=r"Señal ruidosa $y(t)$"
)

plt.title(r"Señal limpia y señal ruidosa, $\sigma_n^2=5$")
plt.xlabel("Tiempo [s]")
plt.ylabel("Amplitud")
plt.grid(True)
plt.xlim(0, 3*T)
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# 12. GRÁFICA SUPERPUESTA DE PSD
# ============================================================

plt.figure(figsize=(14, 5))

plt.plot(
    freqs / f0,
    Pxx_limpia_dB,
    "--",
    linewidth=2,
    label="PSD limpia por FFT"
)

plt.plot(
    freqs / f0,
    Pxx_red_limpia_dB,
    ":",
    linewidth=3,
    label="PSD limpia por red DFT"
)

plt.plot(
    freqs / f0,
    Pyy_promedio_dB,
    linewidth=2,
    alpha=0.75,
    label="PSD promedio con ruido por FFT"
)

plt.plot(
    freqs / f0,
    Pyy_red_dB,
    "-.",
    linewidth=2,
    alpha=0.9,
    label="PSD promedio con ruido por red DFT"
)

plt.title(r"PSD de $x_0(t)$ con $K=9$: FFT clásica vs red DFT")
plt.xlabel(r"Frecuencia normalizada $\omega/\omega_0$")
plt.ylabel("PSD normalizada [dB]")
plt.grid(True)

plt.xlim(-22, 22)
plt.ylim(-120, 10)

plt.xticks(armonicos, labels_armonicos, rotation=45, fontsize=8)

plt.legend()
plt.tight_layout()
plt.show()